# [Pipeline Title] — Explanatory Model

---

## 1. Problem Framing

### Business Problem

<!-- TODO: Describe the business problem in plain language. -->

This pipeline answers the question: **[TODO: one sentence explanatory question]**

The deployed output is a **[TODO: coefficient table / factor report / what-if tool]**
surfaced on [TODO: where in the application].

### Who Cares About This

- **[Role]** — [why they care — what decisions this informs]
- **[Role]** — [why they care]

### Predictive vs. Explanatory

This pipeline uses an **explanatory approach**. The goal is to quantify
relationships between features and the target — not to maximize prediction
accuracy on new data. A defensible, interpretable OLS or Logit model is
preferable here over a black-box ensemble with marginally better predictions.

Coefficients are the primary output. They tell us: holding all other features
constant, what is the estimated change in [target] associated with a one-unit
increase in [feature]?

<!-- TODO: One sentence on the main causal question and why it matters
     operationally. What would the org do differently if they knew the answer? -->

### Success Metrics

- **Primary:** Coefficient significance (p-values), direction, and magnitude
- **Secondary:** [Adjusted R² / Pseudo R² (McFadden)] — model-level fit
- **Not primary:** Predictive accuracy on held-out data — that is the goal of
  the predictive pipeline, not this one

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

# TODO: Import the correct prepare function
# from functions.fn_domain_prep import prepare_residents
# from functions.fn_domain_prep import prepare_donors
# from functions.fn_domain_prep import prepare_social_media

from functions.fn_prepare import define_features, split_data
from functions.fn_model_causal import (
    fit_causal_regression,       # for continuous targets
    fit_causal_classification,   # for binary targets
    get_coefficients,
    check_assumptions,           # OLS only
    check_vif,
    refit_with_robust_se,        # OLS only — apply if homoscedasticity fails
    run_greedy_backward,         # OLS feature selection
    run_stepwise_pvalue,         # Logit feature selection
)

print("All imports successful.")

### 2.1 Load and Prepare Data

`prepare_[domain]()` encodes every cleaning and feature engineering decision
from `eda_[domain].ipynb`. Tries Azure SQL first, falls back to CSV.

**Tables joined:** `[TODO: list tables]`

**Key preparation decisions encoded:**
- [TODO: copy from EDA summary]

In [ ]:
# df, NUMERIC, CATEGORICAL, DROP = prepare_residents()
# df, NUMERIC, CATEGORICAL, DROP = prepare_donors()
# df, NUMERIC, CATEGORICAL, DROP = prepare_social_media()

TARGET = '[TODO: target column name]'

print(f"Dataset shape: {df.shape}")
# Regression:     print(f"Target mean: {df[TARGET].mean():.2f}  |  Std: {df[TARGET].std():.2f}")
# Classification: print(f"Base rate: {df[TARGET].mean():.1%} positive")

### 2.2 Feature Definition

`define_features()` is called with `DROP['[target]']`. Unlike the predictive
pipeline, we do not need `class_weight` or ensemble models here — but leakage
prevention is equally important for causal validity.

In [ ]:
X, y = define_features(
    df,
    target=TARGET,
    numeric=NUMERIC,
    categorical=CATEGORICAL,
    drop_cols=DROP[TARGET],
)

categorical_in_X = [c for c in CATEGORICAL if c in X.columns]
numeric_in_X     = [c for c in NUMERIC     if c in X.columns]
X[categorical_in_X] = X[categorical_in_X].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Numeric:     {len(numeric_in_X)}")
print(f"  Categorical: {len(categorical_in_X)}")

### 2.3 Exploratory Confirmation

EDA was conducted in `eda_[domain].ipynb`. These cells confirm expected
relationships before fitting the explanatory model.

In [ ]:
corr = X[numeric_in_X].corrwith(y).abs().sort_values(ascending=False)
print(f"Top 10 numeric features by |correlation| with {TARGET}:")
print(corr.head(10).round(3).to_string())

In [ ]:
# TODO: Categorical breakdown and distribution plots (same as predictive template)

---
## 3. Causal Model Specification

### 3.1 Train/Test Split

Even for explanatory models we hold out a test set. We do not use it to select
features — that would be leakage — but we use it in Section 4 to give an honest
sense of how well the explanatory model generalizes.

In [ ]:
# Regression: stratify=False | Classification: stratify=True
X_train, X_test, y_train, y_test = split_data(X, y, stratify=False)

### 3.2 Multicollinearity Check (VIF)

Variance Inflation Factor (VIF) measures how much each feature's variance is
inflated by correlation with other features. High VIF (> 10) makes coefficients
unstable and hard to interpret. We drop features iteratively until VIF is
acceptable for all remaining features.

This step is specific to explanatory models — in the predictive pipeline,
multicollinearity is handled implicitly by the preprocessor and regularization.

In [ ]:
# Encode for statsmodels (needs plain numeric matrix)
X_train_enc = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"Encoded matrix: {X_train_enc.shape[0]} rows × {X_train_enc.shape[1]} columns")

# Check VIF — drop features above threshold iteratively
vif_df   = check_vif(X_train_enc, threshold=10.0)
high_vif = vif_df[vif_df['VIF'] > 10]['feature'].tolist() if 'VIF' in vif_df.columns else []
print(f"\nFeatures with VIF > 10: {high_vif if high_vif else 'None'}")

X_clean = X_train_enc.drop(columns=high_vif, errors='ignore')
print(f"\nMatrix after VIF cleanup: {X_clean.shape[1]} features remaining")

### 3.3 Fit Explanatory Model

In [ ]:
# TODO: Choose one:
# Continuous target (OLS):
results = fit_causal_regression(X_clean, y_train)
# Binary target (Logit):
# results = fit_causal_classification(X_clean, y_train.astype(int))

print(results.summary())

### 3.4 OLS Assumption Checks

*(Skip this section for Logit / classification targets — OLS assumptions
do not apply to logistic regression.)*

Five assumptions must hold for OLS coefficients to be unbiased and valid:
1. **Linearity** — the relationship between X and y is linear
2. **Independence** — residuals are not autocorrelated
3. **Normality** — residuals are approximately normally distributed
4. **Homoscedasticity** — residual variance is constant across predicted values
5. **No multicollinearity** — already checked with VIF above

In [ ]:
# OLS only — remove for Logit pipelines
verdicts = check_assumptions(results)

# If homoscedasticity fails, refit with robust standard errors
if verdicts.get('homoscedasticity', {}).get('verdict') != 'PASS':
    print("\n[ACTION] Homoscedasticity failed — refitting with HC3 robust SEs")
    results = refit_with_robust_se(results)
    print(results.summary())

### 3.5 Feature Selection

**OLS (regression):** Greedy backward removal minimizes validation RMSE,
producing the most parsimonious model where every feature earned its place.

**Logit (classification):** Stepwise p-value removal drops the least
significant feature until all remaining features are significant at p < 0.05.

In [ ]:
# ── OLS regression: greedy backward ──────────────────────────────────────────
# from sklearn.model_selection import train_test_split
# X_tr, X_val, y_tr, y_val = train_test_split(X_clean, y_train,
#                                              test_size=0.25, random_state=42)
# trace, optimal_features = run_greedy_backward(
#     X_tr, y_tr, X_val, y_val,
#     numeric_features=numeric_in_X,
#     categorical_features=categorical_in_X,
# )
# X_final = X_clean[optimal_features]
# results_final = fit_causal_regression(X_final, y_train)

# ── Logit classification: stepwise p-value ────────────────────────────────────
# results_final, kept_features = run_stepwise_pvalue(
#     X_clean, y_train.astype(int),
#     p_threshold=0.05, model_type='logistic'
# )

pass  # remove when filled in

---
## 4. Evaluation & Interpretation

### 4.1 Coefficient Table

The primary output of an explanatory pipeline.

In [ ]:
# Linear: model_type='linear' | Logistic: model_type='logistic'
coef_df = get_coefficients(results, model_type='linear')

print("All coefficients (sorted by |coefficient|):")
print(coef_df[['feature', 'coefficient', 'std_err', 'p_value', 'significant']]
      .to_string(index=False))

print("\nSignificant features only (p < 0.05):")
sig = coef_df[coef_df['p_value'] < 0.05].sort_values('coefficient', ascending=False)
print(sig.to_string(index=False))

In [ ]:
# Visualize significant coefficients
if len(sig) > 0:
    sig_plot = sig.set_index('feature')['coefficient'].sort_values()
    colors   = ['coral' if v < 0 else 'steelblue' for v in sig_plot]
    sig_plot.plot(kind='barh', figsize=(10, max(4, len(sig_plot) * 0.4)),
                  color=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title(f'Significant Coefficients — {TARGET}', fontsize=13)
    plt.xlabel('Coefficient')
    plt.tight_layout()
    plt.show()

### 4.2 Model Fit

<!-- TODO: Interpret the model-level fit statistics.

OLS: Report R², Adjusted R², F-statistic and p(F).
  - R²: proportion of variance in the target explained by the model
  - Adjusted R²: R² penalized for number of features — use this to compare
    models with different feature counts
  - F-statistic: tests whether the model is better than a constant

Logit: Report Pseudo R² (McFadden), Log-Likelihood, AIC.
  - McFadden Pseudo R² of 0.2–0.4 is considered good for logistic regression
  - Compare AIC across model specifications — lower is better -->

In [ ]:
print(f"Model fit statistics:")
# OLS:
print(f"  R²:           {results.rsquared:.4f}")
print(f"  Adjusted R²:  {results.rsquared_adj:.4f}")
print(f"  F-statistic:  {results.fvalue:.4f}  (p = {results.f_pvalue:.6f})")
# Logit (uncomment):
# print(f"  Pseudo R² (McFadden): {results.prsquared:.4f}")
# print(f"  Log-Likelihood:       {results.llf:.4f}")
# print(f"  AIC:                  {results.aic:.4f}")

### 4.3 Causal Interpretation

<!-- TODO: Interpret the significant coefficients carefully.

Structure:
1. For each significant feature, state: "A one-unit increase in [feature] is
   associated with a [coefficient] [unit] change in [target], holding all
   other features constant (p = [p-value])."
2. For Logit: convert to odds ratios — "The odds of [positive class] are
   [exp(coef)] times higher per one-unit increase in [feature]."
3. Group into 2-3 thematic clusters
4. Name specific confounders for each cluster
5. State what additional data or study design would be needed to move from
   correlation to causation (e.g., randomized assignment, longitudinal data,
   natural experiment)
6. State one actionable insight the organization can use despite the
   correlation-not-causation limitation -->

### 4.4 Limitations

<!-- TODO: Be explicit about what this model cannot tell us.
- Sample size and how it affects coefficient stability
- Omitted variable bias — what important confounders are not in the data?
- Selection bias — is the observed population representative of the
  population you want to make claims about?
- Temporal validity — do relationships estimated on historical data hold
  going forward? -->

---
## 5. Deployment

The fitted statsmodels results object is not deployed as a prediction endpoint. Instead, the coefficient table informs a **[TODO: static report / what-if tool / dashboard widget]** visible to [TODO: role].

In [ ]:
# Save the coefficient table for use in the application
coef_df.to_csv('models/[TODO: target_name]_coefficients.csv', index=False)
print("Coefficient table saved.")

# Optional: if a predictive version of this model is also deployed as a pkl,
# save it here using save_model() from fn_model_predict.py

---
## 6. API Response Reference

<!-- TODO: Explanatory models are typically deployed differently from predictive
ones. Choose the pattern that fits your use case:

OPTION A — Static coefficient report (no live scoring):
  The coefficient table is computed once, stored as a CSV or JSON, and surfaced
  as a read-only dashboard in the React frontend. No inference server needed.
  The .NET backend serves the CSV directly via a GET endpoint.

OPTION B — What-if tool (interactive):
  The frontend lets staff change input values and see the predicted outcome
  update. This requires a live inference endpoint. Use the predictive pipeline
  template's endpoint pattern — the explanatory model's coefficients inform
  the interpretation text shown alongside the prediction.

OPTION C — Coefficient display on existing pages:
  Surface the top 3-5 significant features and their direction (positive /
  negative) as a "Key Factors" section on the resident case page or donor
  profile. No live scoring needed — just static text generated from the
  coefficient table. -->

---
### Endpoint Function (`endpoints.py`)

<!-- TODO: If deploying as a live endpoint (Option B above), add a function
to `ml-pipelines/inference/endpoints.py`. If deploying as a static report
(Option A or C), this section is not needed. -->

```python
# TODO: Add to endpoints.py if using Option B (what-if tool)
def [TODO: function_name](id_value: int, features: dict, pipeline) -> dict:
    features_df = pd.DataFrame([features])
    # For regression: use predict() not predict_proba()
    prediction  = float(pipeline.predict(features_df)[0])

    return {
        "[id_field]":   id_value,
        "prediction":   round(prediction, 2),
        "model_version": "[TODO: target_name]_v1",
        "predicted_at": datetime.now(timezone.utc).isoformat(),
    }
```

---
### Route to Add (`server.py`)

<!-- TODO: Only needed for Option B. Skip for Options A and C. -->

```python
@app.post("/predict/[TODO: route-name]")
def predict_[TODO: function_name](request: [TODO: Name]Request):
    try:
        pipeline = load_model("[TODO: target_name]")
    except FileNotFoundError as e:
        raise HTTPException(status_code=503, detail=str(e))
    try:
        return [TODO: function_name](
            [TODO: id_field]=request.[TODO: id_field],
            features=request.features,
            pipeline=pipeline,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {e}")
```

---
*Hearth Haven — IS 455 INTEX Pipeline*